# ARC repair mining: four-GPU smoke

This notebook validates the production sharding path. Four independent workers each see one L4, reconstruct the same 16-puzzle global sample, take disjoint modulo shards, screen with batch size 1, and generate failures in batches of up to 4. Only expensive repair failures are materialized; clean solve replay and zero-mask repair examples remain reconstructible from the clean source corpus.


In [ ]:
RUN_EXPERIMENT = True
NUM_PROBES = 16
WORLD_SIZE = 4
ROLLOUT_BATCH_SIZE = 4
SEED = 20260811

MODEL_PATH = '/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'
VALIDATION_PATH = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json'
CODE_DIR = '/kaggle/working/repair_miner_code'
OUTPUT_DIR = '/kaggle/working/repair_multigpu_shards'
FINAL_JSON = '/kaggle/working/repair_multigpu_smoke.json'

print('probes =', NUM_PROBES, 'workers =', WORLD_SIZE, 'rollout batch =', ROLLOUT_BATCH_SIZE)


In [ ]:
if RUN_EXPERIMENT:
    from pathlib import Path
    code_dir = Path(CODE_DIR)
    code_dir.mkdir(parents=True, exist_ok=False)
    (code_dir / 'repair_mining.py').write_text('"""Utilities for mining ARC error-mask repair examples from raw NVARC puzzles.\n\nThe raw NVARC synthetic-puzzle dataset stores one JSON file per underlying\npuzzle.  This module deliberately samples those files rather than treating the\n24/32 augmented SFT records as independent puzzles.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable, Sequence\n\nimport numpy as np\n\n\nARC_TOKENS = list(range(11)) + [15]\nPAD_ID = 13\nEOS_ID = 15\n\n\ndef _stable_u64(text: str) -> int:\n    return int.from_bytes(hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest(), "big")\n\n\ndef validate_grid(grid: Any) -> bool:\n    try:\n        array = np.asarray(grid)\n    except Exception:\n        return False\n    return (\n        array.ndim == 2\n        and 1 <= array.shape[0] <= 30\n        and 1 <= array.shape[1] <= 30\n        and np.issubdtype(array.dtype, np.integer)\n        and bool(np.all((0 <= array) & (array <= 9)))\n    )\n\n\ndef validate_pairs(pairs: Any, minimum: int = 3) -> bool:\n    return (\n        isinstance(pairs, list)\n        and len(pairs) >= minimum\n        and all(\n            isinstance(pair, dict)\n            and validate_grid(pair.get("input"))\n            and validate_grid(pair.get("output"))\n            for pair in pairs\n        )\n    )\n\n\ndef dihedral_transform(grid: Any, transform_id: int) -> np.ndarray:\n    array = np.asarray(grid, dtype=np.int8)\n    if transform_id == 0:\n        return array.copy()\n    if transform_id == 1:\n        return np.rot90(array, 1)\n    if transform_id == 2:\n        return np.rot90(array, 2)\n    if transform_id == 3:\n        return np.rot90(array, 3)\n    if transform_id == 4:\n        return np.fliplr(array)\n    if transform_id == 5:\n        return np.flipud(array)\n    if transform_id == 6:\n        return array.T\n    if transform_id == 7:\n        return np.fliplr(np.rot90(array, 1))\n    raise ValueError(f"Invalid dihedral transform: {transform_id}")\n\n\ndef transform_grid(grid: Any, transform_id: int, color_mapping: Sequence[int]) -> list[list[int]]:\n    if sorted(color_mapping) != list(range(10)):\n        raise ValueError("color_mapping must be a permutation of 0..9")\n    transformed = dihedral_transform(grid, transform_id)\n    return np.asarray(color_mapping, dtype=np.int8)[transformed].tolist()\n\n\n@dataclass(frozen=True)\nclass LeaveOneOutProbe:\n    subset: str\n    puzzle_id: str\n    anchor_id: str\n    source_path: str\n    demonstration_indices: tuple[int, ...]\n    query_index: int\n    transform_id: int\n    color_mapping: tuple[int, ...]\n    demonstrations: tuple[dict[str, Any], ...]\n    query_input: list[list[int]]\n    gold_output: list[list[int]]\n\n    def to_json(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef build_leave_one_out_probe(\n    pairs: list[dict[str, Any]],\n    *,\n    subset: str,\n    puzzle_id: str,\n    anchor_id: str,\n    source_path: str,\n    seed: int,\n    num_demonstrations: int = 2,\n) -> LeaveOneOutProbe:\n    if not validate_pairs(pairs, minimum=num_demonstrations + 1):\n        raise ValueError(f"Need at least {num_demonstrations + 1} valid pairs")\n\n    rng = np.random.default_rng(_stable_u64(f"{seed}:{subset}:{puzzle_id}"))\n    selected = rng.choice(len(pairs), size=num_demonstrations + 1, replace=False).tolist()\n    demonstration_indices = tuple(int(index) for index in selected[:-1])\n    query_index = int(selected[-1])\n    transform_id = int(rng.integers(0, 8))\n    color_mapping = tuple(int(value) for value in rng.permutation(10))\n\n    def transform_pair(index: int) -> dict[str, Any]:\n        pair = pairs[index]\n        return {\n            "input": transform_grid(pair["input"], transform_id, color_mapping),\n            "output": transform_grid(pair["output"], transform_id, color_mapping),\n        }\n\n    demonstrations = tuple(transform_pair(index) for index in demonstration_indices)\n    query_pair = transform_pair(query_index)\n    return LeaveOneOutProbe(\n        subset=subset,\n        puzzle_id=puzzle_id,\n        anchor_id=anchor_id,\n        source_path=source_path,\n        demonstration_indices=demonstration_indices,\n        query_index=query_index,\n        transform_id=transform_id,\n        color_mapping=color_mapping,\n        demonstrations=demonstrations,\n        query_input=query_pair["input"],\n        gold_output=query_pair["output"],\n    )\n\n\ndef load_probe_from_path(\n    path: Path,\n    *,\n    subset: str,\n    seed: int,\n    num_demonstrations: int = 2,\n) -> LeaveOneOutProbe:\n    pairs = json.loads(path.read_text())\n    return build_leave_one_out_probe(\n        pairs,\n        subset=subset,\n        puzzle_id=path.stem,\n        anchor_id=path.parent.name,\n        source_path=str(path),\n        seed=seed,\n        num_demonstrations=num_demonstrations,\n    )\n\n\ndef discover_subset_root(input_root: Path, subset: str) -> Path:\n    # Kaggle mounts this release under its dataset slug.  Prefer the direct\n    # path because recursively walking ~100k small JSON files is needlessly\n    # slow on the read-only FUSE mount.\n    direct_candidates = [\n        input_root / "nvarc-synthetic-puzzles" / subset,\n        input_root / "datasets" / "sorokin" / "nvarc-synthetic-puzzles" / subset,\n    ]\n    for direct in direct_candidates:\n        if direct.is_dir() and any(direct.glob("*/*.json")):\n            return direct\n\n    candidates = []\n    for path in input_root.rglob(subset):\n        if path.is_dir() and any(path.glob("*/*.json")):\n            candidates.append(path)\n    if len(candidates) != 1:\n        raise RuntimeError(f"Expected one raw {subset} root, found: {candidates}")\n    return candidates[0]\n\n\ndef deterministic_sample_paths(\n    subset_root: Path,\n    *,\n    count: int,\n    seed: int,\n    excluded_anchor_ids: Iterable[str] = (),\n) -> list[Path]:\n    """Choose stable files while spreading probes across source-anchor families.\n\n    The raw release has roughly 100k small files.  Enumerating every file on a\n    Kaggle mount is avoidable: rank the much smaller anchor directories, then\n    draw one file per anchor before taking a second file from any anchor.\n    """\n    excluded = set(excluded_anchor_ids)\n    anchors = [path for path in subset_root.iterdir() if path.is_dir() and path.name not in excluded]\n    anchors.sort(key=lambda path: _stable_u64(f"{seed}:anchor:{path.name}"))\n\n    selected = []\n    round_index = 0\n    while len(selected) < count:\n        added = 0\n        for anchor in anchors:\n            files = list(anchor.glob("*.json"))\n            files.sort(key=lambda path: _stable_u64(f"{seed}:file:{path.name}"))\n            if round_index < len(files):\n                selected.append(files[round_index])\n                added += 1\n                if len(selected) == count:\n                    return selected\n        if added == 0:\n            break\n        round_index += 1\n    return selected\n\n\ndef grid_to_string(grid: Any) -> str:\n    if not validate_grid(grid):\n        raise ValueError("Invalid ARC grid")\n    return "\\n".join("".join(str(int(cell)) for cell in row) for row in grid)\n\n\ndef format_prompt(probe: LeaveOneOutProbe) -> str:\n    text = ""\n    for pair in probe.demonstrations:\n        text += (\n            "<|im_start|>user\\n"\n            + grid_to_string(pair["input"])\n            + "<|im_end|><|im_start|>assistant\\n"\n            + grid_to_string(pair["output"])\n            + "<|im_end|>"\n        )\n    return (\n        text\n        + "<|im_start|>user\\n"\n        + grid_to_string(probe.query_input)\n        + "<|im_end|><|im_start|>assistant\\n"\n    )\n\n\ndef format_reply(grid: Any) -> str:\n    return grid_to_string(grid) + "<|im_end|>"\n\n\ndef zero_mask(grid: Any) -> list[list[int]]:\n    if not validate_grid(grid):\n        raise ValueError("Invalid ARC grid")\n    return np.zeros(np.asarray(grid).shape, dtype=np.int8).tolist()\n\n\ndef format_repair_prompt(\n    probe: LeaveOneOutProbe,\n    prediction: Any,\n    *,\n    repair_token: str = "<REPAIR>",\n) -> str:\n    """Append a privileged repair turn to an ordinary solve trajectory.\n\n    The candidate is the model\'s prior assistant response.  The following user\n    turn contains one new mode token and a binary error mask.  The mask is\n    gold-shaped, so its dimensions also communicate the desired output shape.\n    """\n    mask = gold_shape_error_mask(prediction, probe.gold_output)\n    return (\n        format_prompt(probe)\n        + format_reply(prediction)\n        + "<|im_start|>user\\n"\n        + repair_token\n        + "\\n"\n        + grid_to_string(mask)\n        + "<|im_end|><|im_start|>assistant\\n"\n    )\n\n\ndef build_repair_training_record(\n    probe: LeaveOneOutProbe,\n    prediction: Any,\n    *,\n    repair_token: str = "<REPAIR>",\n) -> dict[str, Any]:\n    diagnostics = error_mask_diagnostics(prediction, probe.gold_output)\n    return {\n        "record_type": "repair_noop" if diagnostics["total_wrong_missing_or_extra_cells"] == 0 else "repair_failure",\n        "puzzle_id": probe.puzzle_id,\n        "anchor_id": probe.anchor_id,\n        "source_path": probe.source_path,\n        "demonstration_indices": list(probe.demonstration_indices),\n        "query_index": probe.query_index,\n        "transform_id": probe.transform_id,\n        "color_mapping": list(probe.color_mapping),\n        "input": format_repair_prompt(probe, prediction, repair_token=repair_token),\n        "reply": format_reply(probe.gold_output),\n        "prediction": np.asarray(prediction).tolist(),\n        **diagnostics,\n    }\n\n\ndef build_solve_replay_record(probe: LeaveOneOutProbe) -> dict[str, Any]:\n    """Construct ordinary solve replay without storing a duplicate corpus."""\n    return {\n        "record_type": "solve_replay",\n        "puzzle_id": probe.puzzle_id,\n        "anchor_id": probe.anchor_id,\n        "source_path": probe.source_path,\n        "demonstration_indices": list(probe.demonstration_indices),\n        "query_index": probe.query_index,\n        "transform_id": probe.transform_id,\n        "color_mapping": list(probe.color_mapping),\n        "input": format_prompt(probe),\n        "reply": format_reply(probe.gold_output),\n    }\n\n\ndef length_bucket_batches(\n    values: Sequence[Any],\n    *,\n    batch_size: int,\n    key: Callable[[Any], int],\n) -> list[list[int]]:\n    """Return stable index batches sorted by length to reduce padding."""\n    if batch_size < 1:\n        raise ValueError("batch_size must be positive")\n    ordered = sorted(range(len(values)), key=lambda index: (key(values[index]), index))\n    return [ordered[start : start + batch_size] for start in range(0, len(ordered), batch_size)]\n\n\ndef gold_shape_error_mask(prediction: Any, gold: Any) -> list[list[int]]:\n    """Return a gold-shaped mask; absent cells are wrong and extras are cropped."""\n    if not validate_grid(prediction) or not validate_grid(gold):\n        raise ValueError("prediction and gold must be valid rectangular ARC grids")\n    prediction_array = np.asarray(prediction)\n    gold_array = np.asarray(gold)\n    mask = np.ones(gold_array.shape, dtype=np.int8)\n    rows = min(prediction_array.shape[0], gold_array.shape[0])\n    cols = min(prediction_array.shape[1], gold_array.shape[1])\n    mask[:rows, :cols] = prediction_array[:rows, :cols] != gold_array[:rows, :cols]\n    return mask.tolist()\n\n\ndef error_mask_diagnostics(prediction: Any, gold: Any) -> dict[str, Any]:\n    """Describe cell errors while keeping the repair mask at the gold shape.\n\n    Missing predicted cells are marked in the gold-shaped mask.  Extra cells\n    cannot be represented inside that mask, so they are counted separately;\n    the target shape tells the repair model which suffix rows/columns to drop.\n    """\n    mask = gold_shape_error_mask(prediction, gold)\n    prediction_array = np.asarray(prediction)\n    gold_array = np.asarray(gold)\n    overlap_rows = min(prediction_array.shape[0], gold_array.shape[0])\n    overlap_cols = min(prediction_array.shape[1], gold_array.shape[1])\n    wrong_or_missing = int(np.asarray(mask).sum())\n    extra = int(prediction_array.size - overlap_rows * overlap_cols)\n    return {\n        "error_mask": mask,\n        "prediction_shape": list(prediction_array.shape),\n        "gold_shape": list(gold_array.shape),\n        "shape_equal": prediction_array.shape == gold_array.shape,\n        "wrong_or_missing_gold_cells": wrong_or_missing,\n        "extra_prediction_cells": extra,\n        "total_wrong_missing_or_extra_cells": wrong_or_missing + extra,\n    }\n\n\ndef parse_rollout_grid(tokenizer: Any, token_ids: Sequence[int]) -> tuple[list[list[int]] | None, str | None]:\n    if not token_ids:\n        return None, "empty_rollout"\n    if token_ids[-1] != EOS_ID:\n        return None, "missing_eos"\n    text = tokenizer.decode(list(token_ids[:-1]))\n    try:\n        rows = [[int(character) for character in line] for line in text.strip().split("\\n")]\n    except ValueError:\n        return None, "non_digit_token"\n    if not validate_grid(rows):\n        return None, "malformed_grid"\n    return rows, None\n\n\ndef stabilize_inference_state(model: Any) -> None:\n    """Undo pinned-Unsloth generate() state changes before the next forward.\n\n    In the 2025-09 Kaggle stack, the patched generate path can leave decoder\n    layers with ``gradient_checkpointing=True`` even though their checkpoint\n    callback is absent.  The following ordinary forward then fails.  This is\n    an inference-only pipeline, so checkpointing must remain disabled.\n    """\n    model.eval()\n    disable = getattr(model, "gradient_checkpointing_disable", None)\n    if callable(disable):\n        disable()\n    for module in model.modules():\n        if hasattr(module, "gradient_checkpointing"):\n            module.gradient_checkpointing = False\n\n\ndef teacher_forced_metrics_batch(\n    model: Any,\n    tokenizer: Any,\n    prompts: Sequence[str],\n    gold_replies: Sequence[str],\n) -> list[dict[str, Any]]:\n    import torch\n\n    if len(prompts) != len(gold_replies) or not prompts:\n        raise ValueError("prompts and gold_replies must have the same positive length")\n    stabilize_inference_state(model)\n    prompt_token_ids = [tokenizer.encode(prompt) for prompt in prompts]\n    gold_token_ids = [tokenizer.encode(reply) for reply in gold_replies]\n    if any(not ids for ids in prompt_token_ids) or any(not ids for ids in gold_token_ids):\n        raise ValueError("Prompts and gold replies must tokenize to non-empty sequences")\n\n    sequences = [prompt_ids + gold_ids for prompt_ids, gold_ids in zip(prompt_token_ids, gold_token_ids)]\n    maximum_length = max(map(len, sequences))\n    device = next(model.parameters()).device\n    input_ids = torch.full(\n        (len(sequences), maximum_length),\n        PAD_ID,\n        device=device,\n        dtype=torch.long,\n    )\n    attention_mask = torch.zeros_like(input_ids)\n    for index, sequence in enumerate(sequences):\n        input_ids[index, : len(sequence)] = torch.tensor(sequence, device=device, dtype=torch.long)\n        attention_mask[index, : len(sequence)] = 1\n    with torch.no_grad():\n        logits = model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            return_dict=True,\n            use_cache=False,\n        ).logits\n    legal_ids = torch.tensor(ARC_TOKENS, device=device, dtype=torch.long)\n    results = []\n    for index, (prompt_ids, gold_ids) in enumerate(zip(prompt_token_ids, gold_token_ids)):\n        completion_logits = logits[\n            index,\n            len(prompt_ids) - 1 : len(prompt_ids) - 1 + len(gold_ids),\n        ].float()\n        targets = torch.tensor(gold_ids, device=device, dtype=torch.long)\n        legal_argmax = legal_ids[completion_logits[:, legal_ids].argmax(dim=-1)]\n        log_prob = torch.log_softmax(completion_logits, dim=-1)\n        positions = torch.arange(len(gold_ids), device=device)\n        token_nll = -log_prob[positions, targets]\n        wrong = (legal_argmax != targets).nonzero(as_tuple=False).flatten()\n        results.append(\n            {\n                "prompt_tokens": len(prompt_ids),\n                "gold_tokens": len(gold_ids),\n                "gold_nll": float(token_nll.sum().cpu()),\n                "gold_mean_nll": float(token_nll.mean().cpu()),\n                "restricted_greedy_exact": len(wrong) == 0,\n                "wrong_argmax_tokens": int(len(wrong)),\n                "first_wrong_token": int(wrong[0].cpu()) if len(wrong) else None,\n            }\n        )\n    del input_ids, attention_mask, logits, legal_ids\n    return results\n\n\ndef teacher_forced_metrics(model: Any, tokenizer: Any, prompt: str, gold_reply: str) -> dict[str, Any]:\n    return teacher_forced_metrics_batch(model, tokenizer, [prompt], [gold_reply])[0]\n\n\ndef restricted_greedy_rollout_batch(\n    model: Any,\n    tokenizer: Any,\n    prompts: Sequence[str],\n    *,\n    max_new_tokens: int,\n) -> list[list[int]]:\n    import torch\n    from transformers import LogitsProcessorList\n\n    class ArcOnlyLogitsProcessor:\n        def __call__(self, input_ids, scores):\n            masked = torch.full_like(scores, -torch.inf)\n            masked[:, ARC_TOKENS] = scores[:, ARC_TOKENS]\n            return masked\n\n    if not prompts:\n        raise ValueError("prompts must be non-empty")\n    if max_new_tokens < 1:\n        raise ValueError("max_new_tokens must be positive")\n\n    stabilize_inference_state(model)\n    prompt_token_ids = [tokenizer.encode(prompt) for prompt in prompts]\n    if any(not ids for ids in prompt_token_ids):\n        raise ValueError("Prompts must tokenize to non-empty sequences")\n    maximum_prompt_length = max(map(len, prompt_token_ids))\n    device = next(model.parameters()).device\n    input_ids = torch.full(\n        (len(prompts), maximum_prompt_length),\n        PAD_ID,\n        device=device,\n        dtype=torch.long,\n    )\n    attention_mask = torch.zeros_like(input_ids)\n    for index, prompt_ids in enumerate(prompt_token_ids):\n        offset = maximum_prompt_length - len(prompt_ids)\n        input_ids[index, offset:] = torch.tensor(prompt_ids, device=device, dtype=torch.long)\n        attention_mask[index, offset:] = 1\n    try:\n        with torch.no_grad():\n            generated = model.generate(\n                input_ids=input_ids,\n                attention_mask=attention_mask,\n                do_sample=False,\n                max_new_tokens=max_new_tokens,\n                eos_token_id=EOS_ID,\n                pad_token_id=PAD_ID,\n                logits_processor=LogitsProcessorList([ArcOnlyLogitsProcessor()]),\n                use_cache=True,\n            )\n    finally:\n        stabilize_inference_state(model)\n    results = []\n    for row in generated[:, maximum_prompt_length:].tolist():\n        if EOS_ID in row:\n            row = row[: row.index(EOS_ID) + 1]\n        results.append(row)\n    del input_ids, attention_mask, generated\n    return results\n\n\ndef restricted_greedy_rollout(\n    model: Any,\n    tokenizer: Any,\n    prompt: str,\n    *,\n    max_new_tokens: int,\n) -> list[int]:\n    return restricted_greedy_rollout_batch(\n        model,\n        tokenizer,\n        [prompt],\n        max_new_tokens=max_new_tokens,\n    )[0]\n')
    (code_dir / 'mine_repair_dataset.py').write_text('"""Mine materialized ARC repair failures from the clean raw NVARC pool.\n\nRun one process per GPU.  Every process deterministically reconstructs the same\nglobal path sample, takes its ``global_index % world_size == rank`` shard, and\nwrites only expensive wrong-grid repair records.  Ordinary solve replay and\nzero-mask repair examples are intentionally reconstructed from the original\nclean corpus by the later trainer instead of being duplicated here.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport tempfile\nimport time\nfrom pathlib import Path\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--validation-path", type=Path, required=True)\n    parser.add_argument("--model-path", type=str, required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--num-probes", type=int, required=True)\n    parser.add_argument("--seed", type=int, default=20260811)\n    parser.add_argument("--rank", type=int, default=0)\n    parser.add_argument("--world-size", type=int, default=1)\n    parser.add_argument("--rollout-batch-size", type=int, default=4)\n    parser.add_argument("--max-seq-length", type=int, default=8192)\n    parser.add_argument("--repair-token", type=str, default="<REPAIR>")\n    return parser.parse_args()\n\n\ndef append_jsonl(path: Path, value: dict) -> None:\n    with path.open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps(value, separators=(",", ":")) + "\\n")\n        handle.flush()\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.num_probes < 1:\n        raise ValueError("num_probes must be positive")\n    if not (0 <= args.rank < args.world_size):\n        raise ValueError("rank must be in [0, world_size)")\n    if args.rollout_batch_size < 1:\n        raise ValueError("rollout_batch_size must be positive")\n\n    os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")\n    os.environ.setdefault("HF_HUB_OFFLINE", "1")\n    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\n    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")\n    os.environ.setdefault("TRITON_PTXAS_PATH", "/usr/local/cuda/bin/ptxas")\n    os.environ.setdefault("OMP_NUM_THREADS", "12")\n    compile_root = Path(tempfile.gettempdir()) / f"unsloth_repair_rank{args.rank}_pid{os.getpid()}"\n    compile_root.mkdir(parents=True, exist_ok=True)\n    os.environ["UNSLOTH_COMPILE_LOCATION"] = str(compile_root)\n\n    import unsloth  # noqa: F401 - must precede transformers/Unsloth model imports\n    import torch\n    from unsloth import FastLanguageModel\n\n    from repair_mining import (\n        build_repair_training_record,\n        deterministic_sample_paths,\n        discover_subset_root,\n        format_prompt,\n        format_reply,\n        length_bucket_batches,\n        load_probe_from_path,\n        parse_rollout_grid,\n        restricted_greedy_rollout_batch,\n        stabilize_inference_state,\n        teacher_forced_metrics,\n    )\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    records_path = args.output_dir / f"repair_failures.rank{args.rank}.jsonl"\n    invalid_path = args.output_dir / f"invalid_rollouts.rank{args.rank}.jsonl"\n    summary_path = args.output_dir / f"summary.rank{args.rank}.json"\n    for path in (records_path, invalid_path, summary_path):\n        if path.exists():\n            raise FileExistsError(f"Refusing to overwrite existing output: {path}")\n\n    validation_ids = set(json.loads(args.validation_path.read_text()))\n    training_root = discover_subset_root(args.input_root, "nvarc_training")\n    global_paths = deterministic_sample_paths(\n        training_root,\n        count=args.num_probes,\n        seed=args.seed,\n        excluded_anchor_ids=validation_ids,\n    )\n    if len(global_paths) != args.num_probes:\n        raise RuntimeError(f"Requested {args.num_probes} paths but found {len(global_paths)}")\n\n    indexed_paths = [\n        (global_index, path)\n        for global_index, path in enumerate(global_paths)\n        if global_index % args.world_size == args.rank\n    ]\n    probes = [\n        load_probe_from_path(path, subset="nvarc_training", seed=args.seed)\n        for _, path in indexed_paths\n    ]\n    prompts = [format_prompt(probe) for probe in probes]\n    replies = [format_reply(probe.gold_output) for probe in probes]\n\n    load_started = time.perf_counter()\n    model, tokenizer = FastLanguageModel.from_pretrained(\n        model_name=args.model_path,\n        full_finetuning=False,\n        load_in_4bit=False,\n        local_files_only=True,\n        use_gradient_checkpointing=False,\n        max_seq_length=args.max_seq_length,\n    )\n    model = FastLanguageModel.for_inference(model)\n    stabilize_inference_state(model)\n    if len(tokenizer) != 16:\n        raise RuntimeError(f"Expected the 16-token ARC tokenizer, found {len(tokenizer)}")\n    model_load_seconds = time.perf_counter() - load_started\n\n    metrics = [None] * len(probes)\n    sequence_too_long = []\n    screen_started = time.perf_counter()\n    # Batch 8 was slower than batch 1 in the validated 16-probe benchmark.\n    for index, (prompt, reply) in enumerate(zip(prompts, replies)):\n        prompt_tokens = len(tokenizer.encode(prompt))\n        gold_tokens = len(tokenizer.encode(reply))\n        if prompt_tokens + gold_tokens > args.max_seq_length:\n            sequence_too_long.append(index)\n            continue\n        metrics[index] = teacher_forced_metrics(model, tokenizer, prompt, reply)\n    screen_seconds = time.perf_counter() - screen_started\n\n    failure_indices = [\n        index\n        for index, item in enumerate(metrics)\n        if item is not None and not item["restricted_greedy_exact"]\n    ]\n    failure_prompts = [prompts[index] for index in failure_indices]\n    rollout_batches = length_bucket_batches(\n        failure_prompts,\n        batch_size=args.rollout_batch_size,\n        key=lambda prompt: len(tokenizer.encode(prompt)),\n    )\n\n    counts = {\n        "assigned_probes": len(probes),\n        "sequence_too_long": len(sequence_too_long),\n        "teacher_forced_exact": sum(\n            item is not None and item["restricted_greedy_exact"] for item in metrics\n        ),\n        "teacher_forced_failures": len(failure_indices),\n        "usable_repair_failures": 0,\n        "invalid_rollouts": 0,\n    }\n    rollout_started = time.perf_counter()\n    for batch_number, local_failure_indices in enumerate(rollout_batches, 1):\n        batch_probe_indices = [failure_indices[index] for index in local_failure_indices]\n        batch_prompts = [prompts[index] for index in batch_probe_indices]\n        batch_global_indices = [indexed_paths[index][0] for index in batch_probe_indices]\n        maximum_prompt = max(len(tokenizer.encode(prompt)) for prompt in batch_prompts)\n        token_batches = restricted_greedy_rollout_batch(\n            model,\n            tokenizer,\n            batch_prompts,\n            max_new_tokens=min(930, args.max_seq_length - maximum_prompt),\n        )\n        for probe_index, token_ids in zip(batch_probe_indices, token_batches):\n            global_index, source_path = indexed_paths[probe_index]\n            probe = probes[probe_index]\n            prediction, invalid_reason = parse_rollout_grid(tokenizer, token_ids)\n            if prediction is None:\n                counts["invalid_rollouts"] += 1\n                append_jsonl(\n                    invalid_path,\n                    {\n                        "global_index": global_index,\n                        "puzzle_id": probe.puzzle_id,\n                        "source_relpath": str(source_path.relative_to(training_root)),\n                        "reason": invalid_reason,\n                        "rollout_tokens": len(token_ids),\n                        "teacher_forced": metrics[probe_index],\n                    },\n                )\n                continue\n\n            record = build_repair_training_record(\n                probe,\n                prediction,\n                repair_token=args.repair_token,\n            )\n            record["source_path"] = str(source_path.relative_to(training_root))\n            record.update(\n                {\n                    "global_index": global_index,\n                    "source_relpath": str(source_path.relative_to(training_root)),\n                    "rollout_tokens": len(token_ids),\n                    "teacher_forced": metrics[probe_index],\n                    "decoder": {\n                        "name": "restricted_greedy",\n                        "rollout_batch_size": args.rollout_batch_size,\n                        "batch_number": batch_number,\n                        "batch_member_global_indices": batch_global_indices,\n                        "rank": args.rank,\n                        "world_size": args.world_size,\n                        "max_new_tokens": min(930, args.max_seq_length - maximum_prompt),\n                    },\n                }\n            )\n            append_jsonl(records_path, record)\n            counts["usable_repair_failures"] += 1\n\n        print(\n            f"rank={args.rank} rollout_batch={batch_number}/{len(rollout_batches)} "\n            f"usable={counts[\'usable_repair_failures\']} invalid={counts[\'invalid_rollouts\']}",\n            flush=True,\n        )\n        torch.cuda.empty_cache()\n\n    rollout_seconds = time.perf_counter() - rollout_started\n    summary = {\n        "config": {\n            "num_probes": args.num_probes,\n            "seed": args.seed,\n            "rank": args.rank,\n            "world_size": args.world_size,\n            "rollout_batch_size": args.rollout_batch_size,\n            "max_seq_length": args.max_seq_length,\n            "repair_token": args.repair_token,\n        },\n        "counts": counts,\n        "timings": {\n            "model_load_s": model_load_seconds,\n            "teacher_forced_screen_s": screen_seconds,\n            "rollout_s": rollout_seconds,\n        },\n        "outputs": {\n            "repair_failures": str(records_path),\n            "invalid_rollouts": str(invalid_path) if invalid_path.exists() else None,\n        },\n    }\n    summary_path.write_text(json.dumps(summary, indent=2))\n    print(json.dumps(summary, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
    print('wrote production worker sources')


In [ ]:
if RUN_EXPERIMENT:
    import os, subprocess, sys, time
    from pathlib import Path

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=False)
    processes = []
    handles = []
    started = time.perf_counter()
    for rank in range(WORLD_SIZE):
        log_path = output_dir / f'worker{rank}.log'
        handle = log_path.open('w')
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(rank)
        env['PYTHONUNBUFFERED'] = '1'
        command = [
            sys.executable,
            str(Path(CODE_DIR) / 'mine_repair_dataset.py'),
            '--validation-path', VALIDATION_PATH,
            '--model-path', MODEL_PATH,
            '--output-dir', OUTPUT_DIR,
            '--num-probes', str(NUM_PROBES),
            '--seed', str(SEED),
            '--rank', str(rank),
            '--world-size', str(WORLD_SIZE),
            '--rollout-batch-size', str(ROLLOUT_BATCH_SIZE),
        ]
        processes.append((rank, subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=handle, stderr=subprocess.STDOUT)))
        handles.append(handle)

    failures = []
    for rank, process in processes:
        return_code = process.wait()
        if return_code:
            failures.append((rank, return_code))
    for handle in handles:
        handle.close()
    wall_seconds = time.perf_counter() - started

    if failures:
        for rank, return_code in failures:
            print('FAILED WORKER', rank, 'return code', return_code)
            print((output_dir / f'worker{rank}.log').read_text()[-12000:])
        raise RuntimeError(f'worker failures: {failures}')
    print('all workers complete in', round(wall_seconds, 2), 'seconds')


In [ ]:
if RUN_EXPERIMENT:
    import json
    from pathlib import Path

    output_dir = Path(OUTPUT_DIR)
    summaries = [json.loads((output_dir / f'summary.rank{rank}.json').read_text()) for rank in range(WORLD_SIZE)]
    records = []
    for rank in range(WORLD_SIZE):
        path = output_dir / f'repair_failures.rank{rank}.jsonl'
        if path.exists():
            records.extend(json.loads(line) for line in path.read_text().splitlines() if line.strip())
    records.sort(key=lambda record: record['global_index'])

    aggregate = {
        key: sum(summary['counts'][key] for summary in summaries)
        for key in summaries[0]['counts']
    }
    assert aggregate == {
        'assigned_probes': 16,
        'sequence_too_long': 0,
        'teacher_forced_exact': 3,
        'teacher_forced_failures': 13,
        'usable_repair_failures': 13,
        'invalid_rollouts': 0,
    }, aggregate
    assert len(records) == 13
    assert len({record['global_index'] for record in records}) == 13
    assert all(record['record_type'] == 'repair_failure' for record in records)
    assert all('<REPAIR>\n' in record['input'] for record in records)
    assert all(record['global_index'] in record['decoder']['batch_member_global_indices'] for record in records)
    assert all(len(record['decoder']['batch_member_global_indices']) <= ROLLOUT_BATCH_SIZE for record in records)

    result = {
        'config': {
            'num_probes': NUM_PROBES,
            'world_size': WORLD_SIZE,
            'rollout_batch_size': ROLLOUT_BATCH_SIZE,
            'seed': SEED,
        },
        'wall_seconds': wall_seconds,
        'aggregate_counts': aggregate,
        'worker_summaries': summaries,
        'record_global_indices': [record['global_index'] for record in records],
        'record_schema_example': records[0],
    }
    Path(FINAL_JSON).write_text(json.dumps(result, indent=2))
    print(json.dumps({key: value for key, value in result.items() if key != 'record_schema_example'}, indent=2))
